# `predict_cell_type` tutorial — Basal Ganglia Subclass

End-to-end tutorial for cytozip's methylation cell-type classifier
([`predict_cell_type`](../cytozip/model.py) / [`CellTypeClassifier`](../cytozip/model.py)) on real
**Basal Ganglia (BG)** *Subclass* pseudobulks.

## What the classifier does

Given a **shallow / low-coverage** single cell (few reads per cytosine), assign it to one of a
panel of **deep** cell-type pseudobulk references, using a transparent, training-free
naive-Bayes / methylation-frequency deconvolution:

1. For each candidate type `t` and cytosine `c`, estimate a continuous methylation frequency from
   the pseudobulk with Beta shrinkage $\theta_{c,t} = (m_{c,t}+\alpha_0)/(n_{c,t}+\alpha_0+\beta_0)$.
2. Score a query cell as the aggregated per-cytosine Bernoulli log-likelihood, in two independent
   channels (CpG and CpH).
3. Softmax over types &rarr; calibrated probabilities (supports abstention).

## Pipeline

| stage | what | status |
|-------|------|--------|
| 1. Train | fit per-type CpG/CpH frequencies from the Subclass pseudobulks &rarr; save a reusable model store | **this notebook (run now)** |
| 2. Predict | load the trained model, classify query cells | placeholder — run later |
| 3. Validation | held-out accuracy / macro-F1 / confusion matrix | placeholder — run later |
| 4. Benchmark | speed / memory / downsampling sweeps | placeholder — run later |

> The model is saved so that the later `predict_cell_type(..., outdir=OUTDIR)` call **auto-detects it
> and skips fitting** — no need to re-read the pseudobulks.

## 1. Setup &amp; imports

In [1]:
import os, glob, json, time
import numpy as np
import pandas as pd
import cytozip as czip
from cytozip import CellTypeClassifier, predict_cell_type

print('cytozip', getattr(czip, '__version__', '(dev)'))

cytozip 0.3.9.dev24


## 2. Configuration (inputs, outputs, hyper-parameters)

- **`PSEUDOBULK_DIR`** — the `-s` *source*: a directory of per-cell-type deep pseudobulk `.cz`
  (each file's stem is the cell-type name).
- **`REFERENCE`** — the `-e` *reference*: the `build_ref` allc `.cz` supplying the per-row
  `context` (CpG vs CpH split). All pseudobulks are row-aligned to this axis.
- **`OUTDIR`** — where everything (model now, predictions later) is written. The model store goes
  to `OUTDIR/model` (real disk, so a large memmap never spills into a small/RAM `/tmp` &rarr; SIGBUS).

In [2]:
# --- annotation level to train / predict at -----------------------------
LEVEL = 'Class'   # classification level: 'Subclass' or 'Class'

# --- inputs -------------------------------------------------------------
PSEUDOBULK_DIR = os.path.expanduser(f'~/Projects/BG/pseudobulk/{LEVEL}/cz')       # -s source (per-type .cz)
REFERENCE      = os.path.expanduser('~/Ref/hg38/hg38_with_chrL.allc.cz')         # -e reference (context axis)

# --- outputs ------------------------------------------------------------
OUTDIR    = os.path.expanduser(f'~/Projects/BG/pseudobulk/{LEVEL}/model_{LEVEL}') # Subclass->model_Subclass, Class->model_Class
MODEL_DIR = os.path.join(OUTDIR, 'model')                                        # trained model store (reused by predict)
os.makedirs(OUTDIR, exist_ok=True)

# --- training hyper-parameters -----------------------------------------
LAMBDA_CG = 1.0    # CpG channel log-weight
LAMBDA_CH = 1.0    # CpH channel log-weight (lower it to rebalance if CpH sites dominate)
TOP_CG    = None #0.10   # keep top 10% most discriminative CpG sites (None = keep all)
TOP_CH    = None #0.05   # keep top 5%  most discriminative CpH sites (None = keep all)
N_JOBS    = 72      # parallel .cz readers during fit

for p in (PSEUDOBULK_DIR, REFERENCE):
    assert os.path.exists(p), f'missing input: {p}'
print('level          :', LEVEL)
print('pseudobulk dir :', PSEUDOBULK_DIR)
print('reference      :', REFERENCE)
print('output dir     :', OUTDIR)
print('model store    :', MODEL_DIR)


level          : Class
pseudobulk dir : /home/x-wding2/Projects/BG/pseudobulk/Class/cz
reference      : /home/x-wding2/Ref/hg38/hg38_with_chrL.allc.cz
output dir     : /home/x-wding2/Projects/BG/pseudobulk/Class/model_Class
model store    : /home/x-wding2/Projects/BG/pseudobulk/Class/model_Class/model


## 3. Inspect the pseudobulk inputs

`fit` now accepts the **directory** directly (`pseudobulks=PSEUDOBULK_DIR`) — each `.cz` file's stem
becomes the cell-type label, and non-`.cz` files are ignored. Here we still build the explicit
`{cell_type: path}` mapping just to **preview** the inputs and to align the `cell_counts` below.


In [3]:
cz_files = sorted(glob.glob(os.path.join(PSEUDOBULK_DIR, '*.cz')))
pseudobulks = {os.path.basename(p)[:-3]: p for p in cz_files}   # stem -> path

print(f'{len(pseudobulks)} cell types found:')
for t, p in pseudobulks.items():
    print(f'  {t:<24s} {os.path.getsize(p)/1e6:8.1f} MB')

12 cell types found:
  Astro-Epen                 2411.0 MB
  CN_CGE_GABA                2005.9 MB
  CN_GABA-Glut                787.6 MB
  CN_LGE_GABA                3867.4 MB
  CN_MGE_GABA                2872.8 MB
  Cx_GABA                     545.5 MB
  F_M_GABA                   3065.9 MB
  F_M_Glut                   2615.0 MB
  Immune                     2206.8 MB
  M_Dopa                     1243.5 MB
  OPC-Oligo                  3292.0 MB
  Vascular                   1696.3 MB


## 4. (Optional) abundance prior — `cell_counts`

`cell_counts` records how many cells belong to each Subclass. It does **not** affect the fitted
frequencies; it is stored on the model and only used at **predict** time when `prior_alpha > 0`
(types weighted by $\pi_t \propto \text{cell\_counts}[t]^{\text{prior\_alpha}}$).

The true per-Subclass cell counts come from the atlas annotation table
`~/Projects/BG/clustering/100kb/annotations.tsv` — we take `value_counts()` on its **`Subclass`**
column. Annotation names use spaces while the `.cz` stems use underscores, so we normalise spaces
→ underscores before aligning to our cell types. Prediction still defaults to a **uniform** prior
(`prior_alpha=0`); set `USE_ABUNDANCE_PRIOR = True` to bake the counts into the model.


In [4]:
USE_ABUNDANCE_PRIOR = False   # keep uniform prior by default; set True to bake in the counts

# True per-LEVEL cell counts from the atlas annotation table.
ANNOT_TSV = os.path.expanduser('~/Projects/BG/clustering/100kb/annotations.tsv')

vc = pd.read_csv(ANNOT_TSV, sep='\t', usecols=[LEVEL])[LEVEL].value_counts()
counts_all = {str(name).replace(' ', '_'): int(n) for name, n in vc.items()}   # spaces -> underscores

cell_counts = {t: counts_all[t] for t in pseudobulks if t in counts_all}
missing = [t for t in pseudobulks if t not in counts_all]
print(f'{len(vc)} {LEVEL}s in annotations; matched {len(cell_counts)}/{len(pseudobulks)} pseudobulk types')
if missing:
    print('unmatched pseudobulk types:', missing)
print('example counts:', dict(list(cell_counts.items())[:5]))


12 Classs in annotations; matched 12/12 pseudobulk types
example counts: {'Astro-Epen': 8829, 'CN_CGE_GABA': 2852, 'CN_GABA-Glut': 122, 'CN_LGE_GABA': 145910, 'CN_MGE_GABA': 13254}


## 5. Train — fit the classifier and save the model store

`fit` reads every pseudobulk `.cz` once, estimates the per-type CpG/CpH frequencies, keeps the
`top_cg` / `top_ch` most discriminative sites, and spills the `(n_sites x n_types)` log-likelihood
tables to memory-mappable `.npy` files. Passing `outdir=MODEL_DIR` routes that store **straight to
real disk**; `save(MODEL_DIR)` then writes the `meta.json` header.

This is the same fitting `predict_cell_type` does internally — pre-computing it here means the later
predict/validation/benchmark calls just **load** this model and skip fitting.

In [5]:
t0 = time.time()
clf = CellTypeClassifier(lambda_cg=LAMBDA_CG, lambda_ch=LAMBDA_CH).fit(
    pseudobulks=PSEUDOBULK_DIR,                            # pass the directory directly (stem = cell type)
    reference=REFERENCE,                                   # supplies per-row context (CpG/CpH split)
    cell_counts=(cell_counts if USE_ABUNDANCE_PRIOR else None),
    top_cg=TOP_CG, top_ch=TOP_CH,                          # keep only discriminative sites
    n_jobs=N_JOBS,
    outdir=MODEL_DIR,                                      # memmap store -> real disk (avoids /tmp SIGBUS)
)
clf.save(MODEL_DIR)                                        # write meta.json (arrays already in MODEL_DIR)
print(f'trained {len(clf.cell_types)} cell types in {time.time() - t0:.1f}s')
print('model saved to', MODEL_DIR)


2026-08-04 01:28:44.294 | INFO     | cytozip.model:fit:1007 - CG: Beta prior alpha0=0.7754, beta0=0.1679 (mean=0.822, kappa=0.9433)


2026-08-04 01:28:44.295 | INFO     | cytozip.model:fit:1011 - CH: Beta prior alpha0=0.129, beta0=2.721 (mean=0.045, kappa=2.85)


2026-08-04 01:28:45.231 | INFO     | cytozip.model:fit:1044 - PASS B: scoring cross-type theta range over 12 cell types (58,809,816 CG + 1,141,772,876 CH sites)...


2026-08-04 01:30:31.035 | INFO     | cytozip.model:run:945 - PASS B (theta range): 1/12 cell types done


2026-08-04 01:30:47.012 | INFO     | cytozip.model:run:945 - PASS B (theta range): 2/12 cell types done


...

2026-08-04 02:32:43.967 | INFO     | cytozip.model:fit:1285 - CG: 58,809,816 context sites -> 58,763,396 discriminative sites across 12 cell types in 24 chunks


2026-08-04 02:32:43.968 | INFO     | cytozip.model:fit:1289 - CH: 1,141,772,876 context sites -> 1,141,637,558 discriminative sites across 12 cell types in 24 chunks


2026-08-04 02:32:43.969 | WARNING  | cytozip.model:fit:1298 - model keeps 1,200,400,954 sites x 12 types (~115.2 GB of log-theta). Pass top_cg/top_ch (e.g. top_cg=20000) or min_range_cg/min_range_ch (>0) to select fewer discriminative sites and cut memory drastically.


2026-08-04 02:32:44.083 | INFO     | cytozip.model:save:2285 - saved CellTypeClassifier to /home/x-wding2/Projects/BG/pseudobulk/Class/model_Class/model


trained 12 cell types in 4165.6s
model saved to /home/x-wding2/Projects/BG/pseudobulk/Class/model_Class/model


## 6. Inspect the trained model

Confirm what `fit` selected and the on-disk footprint of the reusable store.

In [5]:
meta = json.load(open(os.path.join(MODEL_DIR, 'meta.json')))
print('cell types      :', len(meta['cell_types']))
print('CpG sites kept  :', meta['cg']['n_sites'] if meta['cg'] else 0)
print('CpH sites kept  :', meta['ch']['n_sites'] if meta['ch'] else 0)
print('alpha0/beta0 CG :', meta['alpha0_cg'], '/', meta['beta0_cg'])
print('alpha0/beta0 CH :', meta['alpha0_ch'], '/', meta['beta0_ch'])

store_size = sum(os.path.getsize(os.path.join(MODEL_DIR, f)) for f in os.listdir(MODEL_DIR))
print('model store size: %.1f MB' % (store_size / 1e6))
print('files           :', sorted(os.listdir(MODEL_DIR)))

cell types      : 12
CpG sites kept  : 58763396
CpH sites kept  : 1141637558
alpha0/beta0 CG : 0.7754489444493425 / 0.16787922268876784
alpha0/beta0 CH : 0.12895926467589316 / 2.721468075056396
model store size: 120040.1 MB
files           : ['cg_log1m_theta.npy', 'cg_log_theta.npy', 'cg_sites.npy', 'ch_log1m_theta.npy', 'ch_log_theta.npy', 'ch_sites.npy', 'meta.json']


---
## 7. Predict for single-cell cz (deep sequencing using snm3C-seq)

**Do not run yet.** Once the query cells are ready, `predict_cell_type` will detect the trained model
under `OUTDIR/model` and **skip fitting**, then write `predictions.csv` + `predict_proba.csv` to
`OUTDIR`. `query` can be a directory of single-cell `.cz`, a concatenated multi-cell `.cz`, a
`{cell_id: path}` dict, or a 2-column `[cell_id, cz_path]` table.

```python
labels, proba = predict_cell_type(
    query=QUERY,               # <- to be provided
    outdir=OUTDIR,             # reuses OUTDIR/model, writes predictions.csv + predict_proba.csv
    reference=REFERENCE,       # only used if the model has to be re-fit
    prior_alpha=0.0,           # uniform prior; raise to use the abundance prior
    abstain_threshold=None,    # e.g. 0.6 to abstain on low-confidence cells
    n_jobs=N_JOBS,
)
```

In [ ]:
# --- query cells: 2000 single-cell .cz ----------------------------------
QUERY_DIR = os.path.expanduser('~/Projects/test_cytozip/benchmark/cz')
assert os.path.isdir(QUERY_DIR), f'missing query dir: {QUERY_DIR}'
n_query = len(glob.glob(os.path.join(QUERY_DIR, '*.cz')))
print(f'{n_query} query .cz cells in {QUERY_DIR}')

# The model under OUTDIR/model is reused (fit is skipped); a directory query
# -> batch predict, returning (labels, proba) and writing them to OUTDIR.
# max_query_cg/ch cap the sites scored PER CELL: deep cells otherwise sum log-lik over
# millions of correlated sites -> naive-Bayes over-accumulates -> confidence saturates at 1.0
# and CN_LGE_GABA gets confidently mis-called as MGE/CGE. Capping ~= MiSeq depth fixes it.
MAX_QUERY_CG = 5000
MAX_QUERY_CH = 10000
t0 = time.time()
labels, proba = predict_cell_type(
    query=QUERY_DIR,
    outdir=OUTDIR,             # reuses OUTDIR/model
    prefix='m3c_ds5k_',        # separate prefix -> keeps the full-depth m3c_*.csv intact
    reference=REFERENCE,       # only used if the model must be re-fit
    prior_alpha=0.0,           # uniform prior
    max_query_cg=MAX_QUERY_CG, max_query_ch=MAX_QUERY_CH,   # cap sites/cell (None = full depth)
    n_jobs=N_JOBS,
)
print(f'predicted {len(labels)} cells in {time.time() - t0:.1f}s')
print('confidence: mean=%.3f  median=%.3f' % (labels['confidence'].mean(),
                                              labels['confidence'].median()))
labels.head()

2026-08-05 10:57:26.728 | INFO     | cytozip.model:predict_cell_type:2557 - found existing model at /home/x-wding2/Projects/BG/pseudobulk/Class/model_Class/model; skipping fit and loading it


2000 query .cz cells in /home/x-wding2/Projects/test_cytozip/benchmark/cz


## 8. Validation

**Do not run yet.** Evaluate the classifier on cells with known labels (e.g. held-out single cells or
downsampled pseudobulks): overall accuracy, **macro-F1** (fair to rare types), and the confusion
matrix. To be filled in when the labelled query set is provided.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, auc)
from sklearn.preprocessing import label_binarize

META_TSV = os.path.expanduser('~/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv')

# read only the cell-id column (first) + LEVEL, keyed by cell id
_id_col = pd.read_csv(META_TSV, sep='\t', nrows=1).columns[0]
meta = pd.read_csv(META_TSV, sep='\t', usecols=[_id_col, LEVEL],
                   dtype=str).set_index(_id_col)
matched = labels.index.isin(meta.index).sum()
print(f"metadata id column: {_id_col!r}; {matched}/{len(labels)} query cells matched")

# true labels aligned to predictions; spaces -> underscores to match model classes
y_true_all = (meta[LEVEL].reindex(labels.index)
              .str.replace(' ', '_', regex=False))
model_classes = list(proba.columns)
mask = y_true_all.notna() & y_true_all.isin(model_classes)
n_out = int((y_true_all.notna() & ~y_true_all.isin(model_classes)).sum())
print(f'{int(mask.sum())}/{len(labels)} cells evaluable '
      f'(labelled & label in the {len(model_classes)} model classes); '
      f'{n_out} labelled but outside model, {int(y_true_all.isna().sum())} unlabelled')

y_true = y_true_all[mask]
y_pred = labels.loc[mask, 'label']
present = [c for c in model_classes if (y_true == c).any()]     # classes actually seen
P = proba.loc[mask, present]
Yb = label_binarize(y_true, classes=present)
Pp = P.values

# ---- headline metrics --------------------------------------------------
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, labels=present, average='macro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, labels=present, average='micro', zero_division=0)
auc_macro = roc_auc_score(Yb, Pp, average='macro')
auc_micro = roc_auc_score(Yb, Pp, average='micro')
ap_macro = average_precision_score(Yb, Pp, average='macro')
ap_micro = average_precision_score(Yb, Pp, average='micro')

print('\n=== overall ===')
print(f'evaluated cells  : {len(y_true)}   classes seen: {len(present)}/{len(model_classes)}')
print(f'accuracy         : {acc:.4f}')
print(f'F1  macro / micro: {f1_macro:.4f} / {f1_micro:.4f}')
print(f'AUC macro / micro: {auc_macro:.4f} / {auc_micro:.4f}')
print(f'AP  macro / micro: {ap_macro:.4f} / {ap_micro:.4f}  (PRC)')

FIG_DIR = os.path.join(OUTDIR, 'validation')
os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
# --- row-normalised confusion matrix ------------------------------------
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
try:
    mpl.rcParams['font.family'] = 'Arial'
except Exception:
    pass

cm = confusion_matrix(y_true, y_pred, labels=present)
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cmn, cmap='Reds', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(present))); ax.set_xticklabels(present, rotation=90, fontsize=14)
ax.set_yticks(range(len(present))); ax.set_yticklabels(present, fontsize=14)
ax.set_xlabel('predicted'); ax.set_ylabel(f'true ({LEVEL})')
ax.set_title(f'Row-normalised confusion matrix (n={len(y_true)}, acc={acc:.3f})')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='fraction of true class')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'm3c_confusion_matrix.pdf'))
plt.show()


In [ ]:
# --- ROC and Precision-Recall curves (one-vs-rest) ----------------------
n_cls = len(present)
fpr, tpr, roc_auc = {}, {}, {}
prec, rec, ap = {}, {}, {}
for i in range(n_cls):
    fpr[i], tpr[i], _ = roc_curve(Yb[:, i], Pp[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    prec[i], rec[i], _ = precision_recall_curve(Yb[:, i], Pp[:, i])
    ap[i] = average_precision_score(Yb[:, i], Pp[:, i])

fpr_mi, tpr_mi, _ = roc_curve(Yb.ravel(), Pp.ravel())          # micro-average
prec_mi, rec_mi, _ = precision_recall_curve(Yb.ravel(), Pp.ravel())
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_cls)]))
mean_tpr = np.mean([np.interp(all_fpr, fpr[i], tpr[i]) for i in range(n_cls)], axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
for i in range(n_cls):
    ax1.plot(fpr[i], tpr[i], lw=0.5, alpha=0.25, color='gray')
ax1.plot(fpr_mi, tpr_mi, lw=2, color='C0', label=f'micro (AUC={auc_micro:.3f})')
ax1.plot(all_fpr, mean_tpr, lw=2, color='C3', label=f'macro (AUC={auc_macro:.3f})')
ax1.plot([0, 1], [0, 1], ls='--', color='k', lw=0.8)
ax1.set_xlabel('false positive rate'); ax1.set_ylabel('true positive rate')
ax1.set_title('ROC (one-vs-rest)'); ax1.legend(loc='lower right', fontsize=9)

for i in range(n_cls):
    ax2.plot(rec[i], prec[i], lw=0.5, alpha=0.25, color='gray')
ax2.plot(rec_mi, prec_mi, lw=2, color='C0', label=f'micro (AP={ap_micro:.3f})')
ax2.axhline(Yb.mean(), ls='--', color='k', lw=0.8, label='chance')
ax2.set_xlabel('recall'); ax2.set_ylabel('precision')
ax2.set_title(f'Precision-Recall (macro AP={ap_macro:.3f})')
ax2.legend(loc='upper right', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'm3c_roc_pr_curves.pdf'))
plt.show()


In [ ]:
# --- per-LEVEL one-vs-rest AUC / AP bars --------------------------------
support = pd.Series(y_true).value_counts().reindex(present).fillna(0).astype(int)
per = pd.DataFrame({
    'AUC': [roc_auc[i] for i in range(n_cls)],
    'AP':  [ap[i] for i in range(n_cls)],
    'support': support.values,
}, index=present).sort_values('AUC', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(per))
ax.bar(x - 0.2, per['AUC'], width=0.4, label='AUC', color='C0')
ax.bar(x + 0.2, per['AP'], width=0.4, label='AP (PRC)', color='C1')
ax.set_xticks(x); ax.set_xticklabels(per.index, rotation=90, fontsize=14)
ax.set_ylim(0, 1.02); ax.set_ylabel('score')
ax.set_title(f'Per-{LEVEL} one-vs-rest AUC and AP')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'm3c_per_class_auc_ap.pdf'))
plt.show()

per.round(4)


## 9. NEXT — Benchmark (run later)

**Do not run yet.** Measure runtime / peak memory and sweep robustness knobs
(`max_query_cg` / `max_query_ch` downsampling, `top_cg` / `top_ch`, `lambda_ch`, `prior_alpha`)
against accuracy. To be filled in after validation.

---
## 10. Predict & validate on MiSeq cells (SALK045)

Same trained model, a different query set: **shallow MiSeq** single cells from `SALK045`. We reuse the
model under `OUTDIR/model` (symlinked into a separate `OUTDIR/miseq_SALK045` so fitting is skipped and
the deep-cell predictions above are not overwritten), classify the MiSeq `.cz`, then score them against
the **same** `META_TSV` Subclass labels and redraw the confusion matrix, ROC/PR curves and per-Subclass
AUC/AP bars.


In [ ]:
# --- MiSeq query cells: SALK045 single-cell .cz -------------------------
MISEQ_QUERY_DIR = os.path.expanduser('~/Projects/test_cytozip/miseq/SALK045/cz/')
assert os.path.isdir(MISEQ_QUERY_DIR), f'missing query dir: {MISEQ_QUERY_DIR}'
n_miseq = len(glob.glob(os.path.join(MISEQ_QUERY_DIR, '*.cz')))
print(f'{n_miseq} MiSeq .cz cells in {MISEQ_QUERY_DIR}')

# separate outdir so MiSeq predictions don't overwrite the deep-cell ones;
# symlink the trained model in so fitting is skipped (model store reused).
MISEQ_OUTDIR = os.path.join(OUTDIR, 'miseq_SALK045')
os.makedirs(MISEQ_OUTDIR, exist_ok=True)
_miseq_model = os.path.join(MISEQ_OUTDIR, 'model')
if not os.path.exists(_miseq_model):
    os.symlink(MODEL_DIR, _miseq_model)

t0 = time.time()
labels_ms, proba_ms = predict_cell_type(
    query=MISEQ_QUERY_DIR,
    outdir=MISEQ_OUTDIR,       # reuses OUTDIR/model
    prefix='miseq_',           # -> MISEQ_OUTDIR/miseq_predictions.csv + miseq_predict_proba.csv
    reference=REFERENCE,       # only used if the model must be re-fit
    prior_alpha=0.0,           # uniform prior
    n_jobs=N_JOBS,
    max_query_cg=MAX_QUERY_CG, max_query_ch=MAX_QUERY_CH,
)
print(f'predicted {len(labels_ms)} MiSeq cells in {time.time() - t0:.1f}s')
print('confidence: mean=%.3f  median=%.3f' % (labels_ms['confidence'].mean(),
                                              labels_ms['confidence'].median()))
labels_ms.head()


In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, auc)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
try:
    mpl.rcParams['font.family'] = 'Arial'
except Exception:
    pass
# --- validate MiSeq predictions against the same META_TSV LEVEL labels ---
META_TSV = os.path.expanduser('~/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv')

_id_col = pd.read_csv(META_TSV, sep='\t', nrows=1).columns[0]
meta = pd.read_csv(META_TSV, sep='\t', usecols=[_id_col, LEVEL],
                   dtype=str).set_index(_id_col)
from cemba_data.mapping.stats import m3c_additional_cols, get_plate_info
plate_info=get_plate_info(meta.index,barcode_version='V2')
for col in plate_info:
    meta[col]=plate_info[col]
meta['real_multiplex_group'] = meta.RandomIndex.apply(
    lambda x: ((int(x[1:]) - 1) % 12) // 2 + 1 if 'unknow' not in x.lower() else 'NA'
)
meta['uid'] = meta.Plate.map(str) + '-' + meta.real_multiplex_group.map(str) + '-' + meta.PCRIndex.map(str)
meta['new_cell_id']=meta.uid.map(str)+'-'+meta.RandomIndex.map(str)
meta.index=meta.new_cell_id.tolist()
matched = labels_ms.index.isin(meta.index).sum()
print(f"metadata id column: {_id_col!r}; {matched}/{len(labels_ms)} MiSeq cells matched")

y_true_all = (meta[LEVEL].reindex(labels_ms.index)
              .str.replace(' ', '_', regex=False))
model_classes = list(proba_ms.columns)
mask = y_true_all.notna() & y_true_all.isin(model_classes)
n_out = int((y_true_all.notna() & ~y_true_all.isin(model_classes)).sum())
print(f'{int(mask.sum())}/{len(labels_ms)} cells evaluable '
      f'(labelled & label in the {len(model_classes)} model classes); '
      f'{n_out} labelled but outside model, {int(y_true_all.isna().sum())} unlabelled')

y_true = y_true_all[mask]
y_pred = labels_ms.loc[mask, 'label']
present = [c for c in model_classes if (y_true == c).any()]
P = proba_ms.loc[mask, present]
Yb = label_binarize(y_true, classes=present)
Pp = P.values

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, labels=present, average='macro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, labels=present, average='micro', zero_division=0)
auc_macro = roc_auc_score(Yb, Pp, average='macro')
auc_micro = roc_auc_score(Yb, Pp, average='micro')
ap_macro = average_precision_score(Yb, Pp, average='macro')
ap_micro = average_precision_score(Yb, Pp, average='micro')

print(f'\n=== MiSeq SALK045 overall ({LEVEL}) ===')
print(f'evaluated cells  : {len(y_true)}   classes seen: {len(present)}/{len(model_classes)}')
print(f'accuracy         : {acc:.4f}')
print(f'F1  macro / micro: {f1_macro:.4f} / {f1_micro:.4f}')
print(f'AUC macro / micro: {auc_macro:.4f} / {auc_micro:.4f}')
print(f'AP  macro / micro: {ap_macro:.4f} / {ap_micro:.4f}  (PRC)')

FIG_DIR = os.path.join(MISEQ_OUTDIR, 'validation')
os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
# --- MiSeq row-normalised confusion matrix ------------------------------
cm = confusion_matrix(y_true, y_pred, labels=present)
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cmn, cmap='Reds', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(present))); ax.set_xticklabels(present, rotation=90, fontsize=14)
ax.set_yticks(range(len(present))); ax.set_yticklabels(present, fontsize=14)
ax.set_xlabel('predicted'); ax.set_ylabel(f'true ({LEVEL})')
ax.set_title(f'MiSeq SALK045 confusion matrix ({LEVEL}, n={len(y_true)}, acc={acc:.3f})')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='fraction of true class')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'miseq_confusion_matrix.pdf'))
plt.show()


In [ ]:
# --- MiSeq ROC and Precision-Recall curves (one-vs-rest) ----------------
n_cls = len(present)
fpr, tpr, roc_auc = {}, {}, {}
prec, rec, ap = {}, {}, {}
for i in range(n_cls):
    fpr[i], tpr[i], _ = roc_curve(Yb[:, i], Pp[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    prec[i], rec[i], _ = precision_recall_curve(Yb[:, i], Pp[:, i])
    ap[i] = average_precision_score(Yb[:, i], Pp[:, i])

fpr_mi, tpr_mi, _ = roc_curve(Yb.ravel(), Pp.ravel())
prec_mi, rec_mi, _ = precision_recall_curve(Yb.ravel(), Pp.ravel())
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_cls)]))
mean_tpr = np.mean([np.interp(all_fpr, fpr[i], tpr[i]) for i in range(n_cls)], axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
for i in range(n_cls):
    ax1.plot(fpr[i], tpr[i], lw=0.5, alpha=0.25, color='gray')
ax1.plot(fpr_mi, tpr_mi, lw=2, color='C0', label=f'micro (AUC={auc_micro:.3f})')
ax1.plot(all_fpr, mean_tpr, lw=2, color='C3', label=f'macro (AUC={auc_macro:.3f})')
ax1.plot([0, 1], [0, 1], ls='--', color='k', lw=0.8)
ax1.set_xlabel('false positive rate'); ax1.set_ylabel('true positive rate')
ax1.set_title('MiSeq ROC (one-vs-rest)'); ax1.legend(loc='lower right', fontsize=9)

for i in range(n_cls):
    ax2.plot(rec[i], prec[i], lw=0.5, alpha=0.25, color='gray')
ax2.plot(rec_mi, prec_mi, lw=2, color='C0', label=f'micro (AP={ap_micro:.3f})')
ax2.axhline(Yb.mean(), ls='--', color='k', lw=0.8, label='chance')
ax2.set_xlabel('recall'); ax2.set_ylabel('precision')
ax2.set_title(f'MiSeq Precision-Recall (macro AP={ap_macro:.3f})')
ax2.legend(loc='upper right', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'miseq_roc_pr_curves.pdf'))
plt.show()


In [ ]:
# --- MiSeq per-LEVEL one-vs-rest AUC / AP bars --------------------------
support = pd.Series(y_true).value_counts().reindex(present).fillna(0).astype(int)
per = pd.DataFrame({
    'AUC': [roc_auc[i] for i in range(n_cls)],
    'AP':  [ap[i] for i in range(n_cls)],
    'support': support.values,
}, index=present).sort_values('AUC', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(per))
ax.bar(x - 0.2, per['AUC'], width=0.4, label='AUC', color='C0')
ax.bar(x + 0.2, per['AP'], width=0.4, label='AP (PRC)', color='C1')
ax.set_xticks(x); ax.set_xticklabels(per.index, rotation=90, fontsize=14)
ax.set_ylim(0, 1.02); ax.set_ylabel('score')
ax.set_title(f'MiSeq SALK045 per-{LEVEL} one-vs-rest AUC and AP')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'miseq_per_class_auc_ap.pdf'))
plt.show()

per.round(4)
